# DATA266 Lab 1 - Task 1: GPT from scratch

This portable notebook works in VSCode/Jupyter or Colab. It uses no cloud-specific imports or personal paths. The model is a character-level GPT implemented from scratch with a causal multi-head attention mask. The config trains for the required 10 epochs and saves resumable checkpoints.

In [ ]:
from pathlib import Path
import subprocess, sys, torch

# When opened directly from GitHub in Colab, the notebook starts in /content
# rather than inside the repository. Clone the current repository automatically.
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'task1_llm').is_dir() and (p / 'requirements.txt').exists()), None)
if REPO_ROOT is None:
    candidate = Path.cwd() / 'GenAI'
    if not (candidate / 'task1_llm').is_dir():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/denishatank12/GenAI.git', str(candidate)], check=True)
    REPO_ROOT = candidate
MEMBER_DIR = REPO_ROOT / 'task1_llm' / 'member_denisha'
print('Repository:', REPO_ROOT)
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (GPU not available)')

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_ROOT / 'requirements.txt')], check=True)
data_path = REPO_ROOT / 'task1_llm' / 'data' / 'TinyStories-train.jsonl'
if not data_path.exists():
    print('TinyStories data is missing; downloading the approved train split.')
    subprocess.run([sys.executable, 'reproducibility/prepare_data.py', '--task', 'task1', '--out', str(REPO_ROOT)], cwd=REPO_ROOT, check=True)
print('Dataset:', data_path, data_path.stat().st_size, 'bytes')
assert data_path.exists() and data_path.stat().st_size > 0

## Smoke test

This must pass before training. It checks forward/backward execution, causal masking, finite gradients, and generation without downloading data or writing results.

In [ ]:
subprocess.run([sys.executable, 'src/smoke_test.py'], cwd=MEMBER_DIR, check=True)

## Train, resume, evaluate, and validate

Run this cell once. If the runtime disconnects, run it again; the latest checkpoint is resumed automatically. All required metrics, curves, samples, failure analysis, raw log, and manifest are produced by the runner.

In [ ]:
subprocess.run([sys.executable, 'run_task1.py'], cwd=MEMBER_DIR, check=True)

In [ ]:
from IPython.display import Image, display
curve = MEMBER_DIR / 'outputs' / 'loss_curves.png'
print((MEMBER_DIR / 'outputs' / 'metrics.csv').read_text())
print('Report:', MEMBER_DIR / 'results.md')
if curve.exists(): display(Image(filename=str(curve)))

## Task 1 submission evidence

Submit/reference these generated artifacts: `member_denisha/results.md`, `failure_analysis.md`, `outputs/metrics.csv`, `outputs/history.json`, `outputs/loss_curves.png`, `outputs/samples.json`, the raw log, the manifest, and the final checkpoint. The failure analysis contains three actual generated-temperature cases.